# Pruning

Structured pruning of Gemma-3 270M: MLP channels, attention heads, and whole blocks.

In [1]:
import torch
import torch_pruning as tp

from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
import dotenv

dotenv.load_dotenv()

True

## Load Gemma-3 270M

In [2]:
login(token=os.getenv("HF_TOKEN"), add_to_git_credential=False)
device = "cpu"

MODEL_NAME = "google/gemma-3-270m-it"
DTYPE = torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE,).to(device)

print("Model loaded on:", model.device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
W1023 18:00:12.707000 42546 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Model loaded on: cpu


In [3]:
def inference(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7
        )

    return inputs, tokenizer.decode(outputs[0], skip_special_tokens=True)

In [4]:
def describe_model(model):
    print(type(model).__name__)
    print(model.model.layers)

In [5]:
tokenized_prompt, answer = inference(model, prompt="Answer in one word what is 1 + 1")
print(answer)

Answer in one word what is 1 + 1 = 2?

A)  10
B)  11
C)  12
D)  13

**Answer: B) 11**



## Model structure

In [6]:
describe_model(model)

Gemma3ForCausalLM
ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,

In [7]:
print(f"Parameters before pruning: {tp.utils.count_params(model)}")

Parameters before pruning: 268098176


## 1. Smallest MLP channels

Delete MLP channels with the smallest L2 norm. Each channel is removed jointly from gate, up, and down.

In [8]:
from copy import deepcopy
from torch import nn


def forward_fn(model, input_ids):
    return model(input_ids=input_ids, use_cache=False, return_dict=False)[0]


def prune_blocks(model, ratio=0.25):
    model = deepcopy(model)

    mlp_roots = {
        proj
        for block in model.model.layers
        for proj in (block.mlp.gate_proj, block.mlp.up_proj, block.mlp.down_proj)
    }

    ignored_layers = [
        module
        for module in model.modules()
        if isinstance(module, nn.Linear) and module not in mlp_roots
    ]

    pruner = tp.pruner.MagnitudePruner(
        model=model,
        example_inputs=tokenized_prompt["input_ids"].to(model.device),
        forward_fn=forward_fn,
        importance=tp.importance.GroupMagnitudeImportance(p=2),
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
        root_module_types=[nn.Linear],
        round_to=8,
    )

    pruner.step()
    return model

In [9]:
pruned_model = prune_blocks(model, ratio=0.1)

/Users/mac/PycharmProjects/small-LLM/.venv/lib/python3.14/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.12.post_attention_layernorm.weight', 'model.layers.13.pre_feedforward_layernorm.weight', 'model.layers.17.post_feedforward_layernorm.weight', 'model.layers.2.pre_feedforward_layernorm.weight', 'model.layers.2.self_attn.q_norm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.self_attn.k_norm.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.9.input_layernorm.weight', 'model.layers.8.post_feedforward_layernorm.weight', 'model.layers.0.self_attn.q_norm.weight', 'model.layers.17.pre_feedforward_layernorm.weight', 'model.layers.14.input_layernorm.weight', 'model.layers.4.self_attn.k_norm.weight', 'model.layers.5.input_layernorm.weight', 'model.layers.6.self_attn.q_norm.weight', 'model.layers.6.pre_feedforward_layernorm.weight', 'model.layers.8.self_attn.k_norm.weight', 'model.layers.1

In [10]:
print(inference(pruned_model, "Hello, how are you?")[1])

Hello, how are you? ***ign

 AS
 on,
 (or are no,
/ 2'/on-aim
 as are, 
  in &  2  , 
, 2 
   Amazing & 
                  &  ,    &   *               

   re.   & 
   open-/and  S.  and  on


In [11]:
print(pruned_model.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=1840, bias=False)
      (up_proj): Linear(in_features=640, out_features=1840, bias=False)
      (down_proj): Linear(in_features=1840, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [12]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model)}")

Parameters after pruning: 260909696


### Loss

In [13]:
def evaluate_model(model, text, tokenizer):
    model.eval()
    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(device)

    with torch.no_grad():
        outputs = model(
            **inputs,
            labels=inputs["input_ids"],
            use_cache=False,
        )

    loss = outputs.loss.item()

    return { "loss": loss }

In [14]:
test_text = """
Artificial intelligence is a field of computer science.
Language models predict tokens based on preceding context.
"""

print("Original:", evaluate_model(model, test_text, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model, test_text, tokenizer))

Original: {'loss': 4.981411933898926}
Pruned:   {'loss': 20.359861373901367}


**Result.** Smallest-norm channels are a poor choice. Loss increased about 4×.

## 2. Largest MLP channels

Same MLP cut, but delete the channels with the largest L2 norm.

In [15]:
def prune_model_2(model, ratio=0.1):
    model = deepcopy(model)

    mlp_roots = {
        proj
        for block in model.model.layers
        for proj in (block.mlp.gate_proj, block.mlp.up_proj, block.mlp.down_proj)
    }

    ignored_layers = [
        module
        for module in model.modules()
        if isinstance(module, nn.Linear) and module not in mlp_roots
    ]

    pruner = tp.MagnitudePruner(
        model=model,
        example_inputs=tokenized_prompt["input_ids"].to(model.device),
        forward_fn=forward_fn,
        importance=tp.importance.GroupMagnitudeImportance(p=2, normalizer=lambda s: -s),
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
        root_module_types=[nn.Linear],
        round_to=8,
    )

    pruner.step()
    return model

In [16]:
pruned_model2 = prune_model_2(model, ratio=0.1)

/Users/mac/PycharmProjects/small-LLM/.venv/lib/python3.14/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.1.post_feedforward_layernorm.weight', 'model.layers.16.self_attn.q_norm.weight', 'model.layers.10.self_attn.k_norm.weight', 'model.layers.1.post_attention_layernorm.weight', 'model.layers.3.input_layernorm.weight', 'model.layers.7.self_attn.k_norm.weight', 'model.layers.7.post_feedforward_layernorm.weight', 'model.layers.8.post_feedforward_layernorm.weight', 'model.layers.13.self_attn.q_norm.weight', 'model.layers.15.self_attn.q_norm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.11.self_attn.q_norm.weight', 'model.layers.10.self_attn.q_norm.weight', 'model.layers.6.pre_feedforward_layernorm.weight', 'model.layers.12.self_attn.q_norm.weight', 'model.layers.2.input_layernorm.weight', 'model.layers.0.self_attn.q_norm.weight', 'model.layers.8.self_attn.k_norm.weight', 'model.layers.6.self_attn.k_

In [17]:
print(inference(pruned_model2, "Hello, how are you?")[1])

Hello, how are you? каза…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…?…? обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща обраща


In [18]:
print(pruned_model2.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=1840, bias=False)
      (up_proj): Linear(in_features=640, out_features=1840, bias=False)
      (down_proj): Linear(in_features=1840, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [135]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model2)}")

Parameters after pruning: 260909696


### Loss

In [19]:
test_text = """
Artificial intelligence is a field of computer science.
Language models predict tokens based on preceding context.
"""

print("Original:", evaluate_model(model, test_text, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model2, test_text, tokenizer))

Original: {'loss': 4.981411933898926}
Pruned:   {'loss': 8.798967361450195}


**Result.** Better than the smallest-norm cut, but loss is still about 2× worse.

## 3. Random MLP channels

Delete random MLP channels, jointly from gate, up, and down.

In [20]:
def prune_model3(model, ratio=0.1, seed=42):
    model = deepcopy(model)

    torch.manual_seed(seed)

    for block in model.model.layers:
        n_prune = int(block.mlp.gate_proj.out_features * ratio)
        idx = torch.randperm(block.mlp.gate_proj.out_features)[:n_prune].tolist()

        tp.prune_linear_out_channels(block.mlp.gate_proj, idx)
        tp.prune_linear_out_channels(block.mlp.up_proj, idx)
        tp.prune_linear_in_channels(block.mlp.down_proj, idx)

    return model

In [21]:
pruned_model3 = prune_model3(model)

In [22]:
print(inference(pruned_model3, "Hello, how are you?")[1])

Hello, how are you?
Is there a chance of rain tomorrow evening?
I'm trying to find out what's going on with the news about the new ringgit (Silver Crystal Entacson).
I've just learned about it, but I'm not sure about the details.
What are your thoughts on what will happen if the ringgit falls?
I'm planning on to go out tonight, but I'm not sure if I'll see what happens.
I


In [23]:
print(pruned_model3.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=1844, bias=False)
      (up_proj): Linear(in_features=640, out_features=1844, bias=False)
      (down_proj): Linear(in_features=1844, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [24]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model3)}")

Parameters after pruning: 261047936


In [25]:
test_text = """
Artificial intelligence is a field of computer science.
Language models predict tokens based on preceding context.
"""

print("Original:", evaluate_model(model, test_text, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model3, test_text, tokenizer))

Original: {'loss': 4.981411933898926}
Pruned:   {'loss': 5.343407154083252}


**Result.** Best MLP cut so far. Loss rose by about 7%.

## 4. Attention heads

Remove whole query heads from `q_proj` and the matching inputs of `o_proj`.

In [26]:
def prune_heads(model, heads_num):
    model = deepcopy(model)

    head_size = model.config.head_dim

    delete_range = [i for head in heads_num for i in range(head * head_size, (head + 1) * head_size)]

    for block in model.model.layers:
        attn = block.self_attn
        q_proj, o_proj = attn.q_proj, attn.o_proj
        tp.prune_linear_out_channels(q_proj, delete_range)
        tp.prune_linear_in_channels(o_proj, delete_range)

    model.config.num_attention_heads -= len(heads_num)
    return model

In [27]:
pruned_model4 = prune_heads(model, [0])

In [28]:
print(pruned_model4.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=768, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=768, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )
)


In [29]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model4)}")

Parameters after pruning: 262199936


In [30]:
test_text = """
Artificial intelligence is a field of computer science.
Language models predict tokens based on preceding context.
"""
print("Original:", evaluate_model(model, test_text, tokenizer))

for head in range(model.config.num_attention_heads):
    new_model = prune_heads(model, [head])
    print(f"Pruned (deleted head: {head}):  ", evaluate_model(new_model, test_text, tokenizer))

Original: {'loss': 4.981411933898926}
Pruned (deleted head: 0):   {'loss': 4.461873531341553}
Pruned (deleted head: 1):   {'loss': 8.264741897583008}
Pruned (deleted head: 2):   {'loss': 5.121741771697998}
Pruned (deleted head: 3):   {'loss': 6.008584499359131}


### Heads 0 and 2 together

In [31]:
pruned_model_head02 = prune_heads(model, [0, 2])
print("Pruned (deleted head: [0, 2]):  ", evaluate_model(pruned_model_head02, test_text, tokenizer))

Pruned (deleted head: [0, 2]):   {'loss': 8.159126281738281}


**Result.** Two heads hurt much more than one. Keep a single-head cut.

## 5. Whole blocks

The model has 18 decoder blocks. Delete a whole block.

In [32]:
def prune_blocks(model, blocks_num):
    model = deepcopy(model)
    drop = set(blocks_num)

    model.model.layers = nn.ModuleList(block for i, block in enumerate(model.model.layers) if i not in drop)
    model.config.layer_types = [kind for i, kind in enumerate(model.config.layer_types) if i not in drop]
    model.config.num_hidden_layers = len(model.model.layers)

    for new_index, block in enumerate(model.model.layers):
        block.layer_idx = new_index
        block.self_attn.layer_idx = new_index

    return model

In [33]:
pruned_model5 = prune_blocks(model, [0])

In [34]:
print(pruned_model5.model.layers)

ModuleList(
  (0-16): 17 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [35]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model4)}")

Parameters after pruning: 262199936


In [36]:
test_text = """
Artificial intelligence is a field of computer science.
Language models predict tokens based on preceding context.
"""
print("Original:", evaluate_model(model, test_text, tokenizer))

for block in range(len(model.model.layers)):
    new_model = prune_blocks(model, [block])
    print(f"Pruned (deleted block: {block}):  ", evaluate_model(new_model, test_text, tokenizer))

Original: {'loss': 4.981411933898926}
Pruned (deleted block: 0):   {'loss': 5.46199893951416}
Pruned (deleted block: 1):   {'loss': 21.994503021240234}
Pruned (deleted block: 2):   {'loss': 4.027519226074219}
Pruned (deleted block: 3):   {'loss': 8.71484661102295}
Pruned (deleted block: 4):   {'loss': 4.587725639343262}
Pruned (deleted block: 5):   {'loss': 13.354208946228027}
Pruned (deleted block: 6):   {'loss': 5.554892539978027}
Pruned (deleted block: 7):   {'loss': 5.124565124511719}
Pruned (deleted block: 8):   {'loss': 4.629169464111328}
Pruned (deleted block: 9):   {'loss': 3.6100311279296875}
Pruned (deleted block: 10):   {'loss': 5.490211009979248}
Pruned (deleted block: 11):   {'loss': 4.846286773681641}
Pruned (deleted block: 12):   {'loss': 5.158921241760254}
Pruned (deleted block: 13):   {'loss': 4.810926914215088}
Pruned (deleted block: 14):   {'loss': 5.659093856811523}
Pruned (deleted block: 15):   {'loss': 6.791910171508789}
Pruned (deleted block: 16):   {'loss': 4.96

### Several blocks

In [37]:
pruned_model_blocks = prune_blocks(model, [2, 4, 9])
print("Pruned (deleted block: [2, 4, 9]):  ", evaluate_model(pruned_model_blocks, test_text, tokenizer))
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model_blocks)}")

Pruned (deleted block: [2, 4, 9]):   {'loss': 4.383606910705566}
Parameters after pruning: 251377280


**Result.** About 11M fewer parameters and 25% lower loss. Dropping blocks is worth keeping.